# Grid-Level Pressure Distribution EDA

This notebook visualizes pressure distributions in LDAPS and GFS by grid.

Scope:
- Source data: train LDAPS and GFS files.
- Pressure variables: `surface_0_sp`, `meanSea_0_prmsl`.
- Unit conversion: Pa to hPa for readable plots.
- Groupings:
  - Month x hour of day.
  - Season x hour of day.
- Outputs:
  - Summary CSV files with count, mean, std, median, quartiles, min, and max.
  - Grid-panel heatmaps for mean pressure and pressure variability.

In [1]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["font.family"] = ["Malgun Gothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / "train"
OUTPUT_DIR = BASE_DIR / "outputs" / "pressure_grid_eda"
PLOT_DIR = OUTPUT_DIR / "plots"
for path in [OUTPUT_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LDAPS_PATH = TRAIN_DIR / "ldaps_train.csv"
GFS_PATH = TRAIN_DIR / "gfs_train.csv"

PRESSURE_COLS = ["surface_0_sp", "meanSea_0_prmsl"]
SOURCE_PATHS = {
    "ldaps": LDAPS_PATH,
    "gfs": GFS_PATH,
}
SEASON_ORDER = ["Winter", "Spring", "Summer", "Autumn"]
SEASON_MONTHS = {
    "Winter": [12, 1, 2],
    "Spring": [3, 4, 5],
    "Summer": [6, 7, 8],
    "Autumn": [9, 10, 11],
}
MONTH_LABELS = [f"{m:02d}" for m in range(1, 13)]
HOUR_LABELS = list(range(24))

OUTPUT_PATHS = {
    "pressure_long": OUTPUT_DIR / "pressure_grid_hourly_long.csv",
    "month_hour_summary": OUTPUT_DIR / "pressure_month_hour_grid_summary.csv",
    "season_hour_summary": OUTPUT_DIR / "pressure_season_hour_grid_summary.csv",
    "overall_grid_summary": OUTPUT_DIR / "pressure_overall_grid_summary.csv",
}

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PLOT_DIR:", PLOT_DIR)

BASE_DIR: C:\Users\심현석\Documents\BARAM 2026
OUTPUT_DIR: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda
PLOT_DIR: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots


## 1. Load Pressure Variables

In [2]:
def month_to_season(month: int) -> str:
    for season, months in SEASON_MONTHS.items():
        if month in months:
            return season
    raise ValueError(f"Invalid month: {month}")


def load_pressure_data(path: Path, source: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    usecols = ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"] + PRESSURE_COLS
    df = pd.read_csv(path, usecols=usecols, parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"])
    df = df.sort_values(["forecast_kst_dtm", "grid_id", "data_available_kst_dtm"]).drop_duplicates(["forecast_kst_dtm", "grid_id"], keep="last")
    df["source"] = source
    df["month"] = df["forecast_kst_dtm"].dt.month
    df["month_label"] = df["month"].map(lambda x: f"{x:02d}")
    df["hour"] = df["forecast_kst_dtm"].dt.hour
    df["season"] = pd.Categorical(df["month"].map(month_to_season), categories=SEASON_ORDER, ordered=True)

    long = df.melt(
        id_vars=["source", "forecast_kst_dtm", "grid_id", "latitude", "longitude", "month", "month_label", "hour", "season"],
        value_vars=PRESSURE_COLS,
        var_name="variable",
        value_name="pressure_pa",
    )
    long["pressure_hpa"] = long["pressure_pa"] / 100.0
    return long


pressure_parts = []
for source, path in SOURCE_PATHS.items():
    part = load_pressure_data(path, source)
    pressure_parts.append(part)
    print(source, part.shape, part["forecast_kst_dtm"].min(), "~", part["forecast_kst_dtm"].max(), "grids=", part["grid_id"].nunique())

pressure_long = pd.concat(pressure_parts, ignore_index=True)
pressure_long.to_csv(OUTPUT_PATHS["pressure_long"], index=False, encoding="utf-8-sig")

print("pressure_long:", pressure_long.shape)
display(pressure_long.head())
display(pressure_long.groupby(["source", "variable"])["pressure_hpa"].describe().round(3))

ldaps (841728, 12) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00 grids= 16


gfs (473472, 12) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00 grids= 9


pressure_long: (1315200, 12)


,source,forecast_kst_dtm,grid_id,latitude,longitude,month,month_label,hour,season,variable,pressure_pa,pressure_hpa
0,ldaps,2022-01-01 01:00:00,1,37.3032,128.9443,1,01,1,Winter,surface_0_sp,90628.91,906.2891
1,ldaps,2022-01-01 01:00:00,2,37.3027,128.9617,1,01,1,Winter,surface_0_sp,91273.41,912.7341
2,ldaps,2022-01-01 01:00:00,3,37.3022,128.9790,1,01,1,Winter,surface_0_sp,92078.41,920.7841
3,ldaps,2022-01-01 01:00:00,4,37.2899,128.9263,1,01,1,Winter,surface_0_sp,91446.91,914.4691
4,ldaps,2022-01-01 01:00:00,5,37.2894,128.9437,1,01,1,Winter,surface_0_sp,90582.91,905.8291


count      mean     std      min       25%       50%       75%       max
source variable                                                                                    
gfs    meanSea_0_prmsl  236736.0  1016.354   7.850  984.400  1010.180  1016.595  1022.341  1038.585
       surface_0_sp     236736.0   954.441  27.456  905.698   931.981   949.485   968.665  1033.443
ldaps  meanSea_0_prmsl  420864.0  1014.543   7.608  980.390  1008.696  1015.143  1020.269  1033.951
       surface_0_sp     420864.0   906.735   7.065  874.628   901.882   906.621   911.530   930.831

## 2. Build Monthly-Hourly And Seasonal-Hourly Summaries

In [3]:
def q25(x: pd.Series) -> float:
    return x.quantile(0.25)


def q75(x: pd.Series) -> float:
    return x.quantile(0.75)


def summarize_pressure(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    return (
        df.groupby(group_cols, observed=True)["pressure_hpa"]
        .agg(
            count="count",
            mean_hpa="mean",
            std_hpa="std",
            median_hpa="median",
            q25_hpa=q25,
            q75_hpa=q75,
            min_hpa="min",
            max_hpa="max",
        )
        .reset_index()
    )


month_hour_summary = summarize_pressure(
    pressure_long,
    ["source", "variable", "grid_id", "latitude", "longitude", "month", "month_label", "hour"],
)
season_hour_summary = summarize_pressure(
    pressure_long,
    ["source", "variable", "grid_id", "latitude", "longitude", "season", "hour"],
)
overall_grid_summary = summarize_pressure(
    pressure_long,
    ["source", "variable", "grid_id", "latitude", "longitude"],
)

month_hour_summary.to_csv(OUTPUT_PATHS["month_hour_summary"], index=False, encoding="utf-8-sig")
season_hour_summary.to_csv(OUTPUT_PATHS["season_hour_summary"], index=False, encoding="utf-8-sig")
overall_grid_summary.to_csv(OUTPUT_PATHS["overall_grid_summary"], index=False, encoding="utf-8-sig")

print("month_hour_summary:", month_hour_summary.shape)
print("season_hour_summary:", season_hour_summary.shape)
print("overall_grid_summary:", overall_grid_summary.shape)
display(month_hour_summary.head())
display(season_hour_summary.head())
display(overall_grid_summary.sort_values(["source", "variable", "grid_id"]).head(20))

month_hour_summary: (14400, 16)
season_hour_summary: (4800, 15)
overall_grid_summary: (50, 13)


,source,variable,grid_id,latitude,longitude,month,month_label,hour,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
0,gfs,meanSea_0_prmsl,1,37.5,128.75,1,01,0,93,1025.330865,4.571904,1025.4585,1022.22260,1029.1270,1013.18710,1033.44980
1,gfs,meanSea_0_prmsl,1,37.5,128.75,1,01,1,93,1024.950916,4.753075,1025.3132,1021.48430,1028.6404,1012.55400,1033.20625
2,gfs,meanSea_0_prmsl,1,37.5,128.75,1,01,2,93,1025.086751,4.765694,1025.3284,1021.70940,1028.3983,1012.95200,1033.38880
3,gfs,meanSea_0_prmsl,1,37.5,128.75,1,01,3,93,1025.063854,4.818052,1025.1540,1021.66664,1028.8825,1013.09914,1033.36330
4,gfs,meanSea_0_prmsl,1,37.5,128.75,1,01,4,93,1024.821554,4.807463,1024.8867,1021.39580,1028.9297,1012.68370,1033.16766


,source,variable,grid_id,latitude,longitude,season,hour,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
0,gfs,meanSea_0_prmsl,1,37.5,128.75,Winter,0,271,1025.245446,5.085188,1025.51140,1022.32375,1028.859650,1006.54766,1036.94375
1,gfs,meanSea_0_prmsl,1,37.5,128.75,Winter,1,271,1024.878031,5.238325,1025.36086,1021.81440,1028.454650,1006.29470,1037.01710
2,gfs,meanSea_0_prmsl,1,37.5,128.75,Winter,2,271,1024.957562,5.255680,1025.40766,1021.90205,1028.515825,1006.41210,1037.24310
3,gfs,meanSea_0_prmsl,1,37.5,128.75,Winter,3,271,1024.916105,5.242218,1025.23110,1021.76100,1028.440250,1006.53580,1037.40625
4,gfs,meanSea_0_prmsl,1,37.5,128.75,Winter,4,271,1024.696266,5.255104,1024.93290,1021.60555,1028.191025,1006.49440,1036.68270


,source,variable,grid_id,latitude,longitude,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
0,gfs,meanSea_0_prmsl,1,37.5000,128.7500,26304,1016.723842,7.907414,1016.857030,1010.466330,1022.754075,988.80330,1037.95734
1,gfs,meanSea_0_prmsl,2,37.5000,129.0000,26304,1016.164269,7.897383,1016.436850,1009.966650,1022.196400,989.63330,1036.24550
2,gfs,meanSea_0_prmsl,3,37.5000,129.2500,26304,1015.527474,7.920621,1015.940580,1009.271150,1021.596463,989.03516,1035.59234
3,gfs,meanSea_0_prmsl,4,37.2500,128.7500,26304,1016.960836,7.913696,1017.093950,1010.642330,1023.086065,987.15330,1038.58490
4,gfs,meanSea_0_prmsl,5,37.2500,129.0000,26304,1016.515981,7.734845,1016.688825,1010.407665,1022.389575,988.85920,1036.70940
5,gfs,meanSea_0_prmsl,6,37.2500,129.2500,26304,1015.927255,7.798460,1016.276850,1009.794350,1021.910112,989.19290,1036.11140
6,gfs,meanSea_0_prmsl,7,37.0000,128.7500,26304,1016.714380,7.871583,1016.932250,1010.464575,1022.835575,986.71440,1037.82140
7,gfs,meanSea_0_prmsl,8,37.0000,129.0000,26304,1016.587479,7.742215,1016.792500,1010.489500,1022.542465,985.67640,1037.11734
8,gfs,meanSea_0_prmsl,9,37.0000,129.2500,26304,1016.062465,7.753653,1016.368700,1009.960415,1021.983025,984.40040,1036.67140
9,gfs,surface_0_sp,1,37.5000,128.7500,26304,929.353157,5.555763,929.725750,925.336100,933.239810,906.88234,945.58960


## 3. Plot Helpers

In [4]:
def safe_filename(text: str, max_len: int = 180) -> str:
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe[:max_len]


def subplot_shape(n_panels: int, source: str) -> tuple[int, int]:
    if source == "ldaps":
        return 4, 4
    if source == "gfs":
        return 3, 3
    ncols = min(4, math.ceil(math.sqrt(n_panels)))
    nrows = math.ceil(n_panels / ncols)
    return nrows, ncols


def metric_label(metric: str) -> str:
    labels = {
        "mean_hpa": "mean pressure (hPa)",
        "std_hpa": "pressure std (hPa)",
        "median_hpa": "median pressure (hPa)",
        "q25_hpa": "pressure q25 (hPa)",
        "q75_hpa": "pressure q75 (hPa)",
    }
    return labels.get(metric, metric)


def plot_grid_panel_heatmaps(
    summary_df: pd.DataFrame,
    source: str,
    variable: str,
    grouping: str,
    metric: str,
    cmap: str,
) -> Path | None:
    data = summary_df[(summary_df["source"].eq(source)) & (summary_df["variable"].eq(variable))].copy()
    if data.empty:
        return None

    grid_ids = sorted(data["grid_id"].dropna().unique())
    nrows, ncols = subplot_shape(len(grid_ids), source)
    if grouping == "month_hour":
        y_col = "month_label"
        y_order = MONTH_LABELS
        title_group = "month x hour"
        fig_height = 2.35 * nrows
    elif grouping == "season_hour":
        y_col = "season"
        y_order = SEASON_ORDER
        title_group = "season x hour"
        fig_height = 2.0 * nrows
    else:
        raise ValueError(grouping)

    vmin = data[metric].min()
    vmax = data[metric].max()
    if metric.endswith("std_hpa"):
        vmin = 0

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.25 * ncols, fig_height), squeeze=False)
    axes_flat = axes.ravel()

    for ax, grid_id in zip(axes_flat, grid_ids):
        grid_df = data[data["grid_id"].eq(grid_id)].copy()
        pivot = grid_df.pivot(index=y_col, columns="hour", values=metric)
        pivot = pivot.reindex(index=y_order, columns=HOUR_LABELS)
        sns.heatmap(
            pivot,
            ax=ax,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            cbar=False,
            linewidths=0.08,
            linecolor="white",
            xticklabels=4,
            yticklabels=True,
        )
        ax.set_title(f"grid {int(grid_id)}", fontsize=10)
        ax.set_xlabel("hour")
        ax.set_ylabel("month" if grouping == "month_hour" else "season")
        ax.tick_params(axis="x", labelrotation=0)
        ax.tick_params(axis="y", labelrotation=0)

    for ax in axes_flat[len(grid_ids):]:
        ax.axis("off")

    fig.subplots_adjust(left=0.055, right=0.885, top=0.91, bottom=0.075, wspace=0.25, hspace=0.62)
    cbar_ax = fig.add_axes([0.91, 0.19, 0.016, 0.62])
    mappable = axes_flat[0].collections[0]
    cbar = fig.colorbar(mappable, cax=cbar_ax)
    cbar.set_label(metric_label(metric))
    fig.suptitle(f"{source.upper()} {variable}: {metric_label(metric)} by {title_group} and grid", y=0.985, fontsize=14)

    out = PLOT_DIR / f"pressure_{source}_{safe_filename(variable)}_{grouping}_{metric}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out

## 4. Save Monthly-Hourly And Seasonal-Hourly Pressure Heatmaps

In [5]:
plot_files = []
plot_specs = [
    ("month_hour", month_hour_summary),
    ("season_hour", season_hour_summary),
]
metrics = [
    ("mean_hpa", "YlGnBu"),
    ("std_hpa", "YlOrRd"),
]

for grouping, summary_df in plot_specs:
    for source in sorted(summary_df["source"].unique()):
        for variable in PRESSURE_COLS:
            for metric, cmap in metrics:
                out = plot_grid_panel_heatmaps(summary_df, source, variable, grouping, metric, cmap)
                if out is not None:
                    plot_files.append(out)

print("Saved plot files:", len(plot_files))
for path in plot_files:
    print("-", path)

Saved plot files: 16
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_gfs_surface_0_sp_month_hour_mean_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_gfs_surface_0_sp_month_hour_std_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_gfs_meanSea_0_prmsl_month_hour_mean_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_gfs_meanSea_0_prmsl_month_hour_std_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_ldaps_surface_0_sp_month_hour_mean_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_ldaps_surface_0_sp_month_hour_std_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_ldaps_meanSea_0_prmsl_month_hour_mean_hpa.png
- C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots\pressure_ldaps_meanSea_0_prmsl_month_hour_std_hpa.png
- C:\Users\심현석\Docu

## 5. Quick Tables For Interpretation

In [6]:
print("Overall grid pressure summary, sorted by source/variable/grid:")
display(overall_grid_summary.sort_values(["source", "variable", "grid_id"]))

print("Highest mean pressure month-hour combinations:")
display(month_hour_summary.sort_values("mean_hpa", ascending=False).head(20))

print("Lowest mean pressure month-hour combinations:")
display(month_hour_summary.sort_values("mean_hpa", ascending=True).head(20))

print("Highest pressure variability season-hour combinations:")
display(season_hour_summary.sort_values("std_hpa", ascending=False).head(20))

print("Output files:")
for name, path in OUTPUT_PATHS.items():
    print(f"- {name}: {path}")
print("Plot directory:", PLOT_DIR)

Overall grid pressure summary, sorted by source/variable/grid:


,source,variable,grid_id,latitude,longitude,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
0,gfs,meanSea_0_prmsl,1,37.5000,128.7500,26304,1016.723842,7.907414,1016.857030,1010.466330,1022.754075,988.80330,1037.95734
1,gfs,meanSea_0_prmsl,2,37.5000,129.0000,26304,1016.164269,7.897383,1016.436850,1009.966650,1022.196400,989.63330,1036.24550
2,gfs,meanSea_0_prmsl,3,37.5000,129.2500,26304,1015.527474,7.920621,1015.940580,1009.271150,1021.596463,989.03516,1035.59234
3,gfs,meanSea_0_prmsl,4,37.2500,128.7500,26304,1016.960836,7.913696,1017.093950,1010.642330,1023.086065,987.15330,1038.58490
4,gfs,meanSea_0_prmsl,5,37.2500,129.0000,26304,1016.515981,7.734845,1016.688825,1010.407665,1022.389575,988.85920,1036.70940
5,gfs,meanSea_0_prmsl,6,37.2500,129.2500,26304,1015.927255,7.798460,1016.276850,1009.794350,1021.910112,989.19290,1036.11140
6,gfs,meanSea_0_prmsl,7,37.0000,128.7500,26304,1016.714380,7.871583,1016.932250,1010.464575,1022.835575,986.71440,1037.82140
7,gfs,meanSea_0_prmsl,8,37.0000,129.0000,26304,1016.587479,7.742215,1016.792500,1010.489500,1022.542465,985.67640,1037.11734
8,gfs,meanSea_0_prmsl,9,37.0000,129.2500,26304,1016.062465,7.753653,1016.368700,1009.960415,1021.983025,984.40040,1036.67140
9,gfs,surface_0_sp,1,37.5000,128.7500,26304,929.353157,5.555763,929.725750,925.336100,933.239810,906.88234,945.58960


Highest mean pressure month-hour combinations:


,source,variable,grid_id,latitude,longitude,month,month_label,hour,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
911,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,23,85,1026.479809,4.408626,1026.70914,1024.05020,1028.80700,1013.26530,1037.38150
896,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,8,85,1026.459917,4.875706,1025.78020,1023.84810,1029.66266,1010.53190,1037.34550
888,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,0,85,1026.443267,4.467123,1026.79960,1023.92200,1028.53520,1014.83695,1037.20945
910,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,22,85,1026.388671,4.436278,1026.64850,1023.87150,1028.90710,1012.44305,1037.12980
47,gfs,meanSea_0_prmsl,1,37.50,128.75,2,02,23,85,1026.254938,4.510221,1026.24360,1023.54420,1028.84650,1012.60130,1037.11550
895,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,7,85,1026.228315,4.870881,1026.03414,1023.57060,1029.40420,1010.68875,1037.10650
32,gfs,meanSea_0_prmsl,1,37.50,128.75,2,02,8,85,1026.220158,4.938997,1025.58550,1023.43234,1029.66640,1010.60790,1036.94140
24,gfs,meanSea_0_prmsl,1,37.50,128.75,2,02,0,85,1026.209213,4.564267,1026.37516,1023.51000,1028.60300,1014.01650,1036.92740
889,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,1,85,1026.208134,4.599175,1026.18670,1023.78016,1028.84875,1014.20330,1036.68730
890,gfs,meanSea_0_prmsl,4,37.25,128.75,2,02,2,85,1026.205777,4.653104,1026.38695,1023.66950,1029.01360,1014.15550,1036.75580


Lowest mean pressure month-hour combinations:


,source,variable,grid_id,latitude,longitude,month,month_label,hour,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
11368,ldaps,surface_0_sp,6,37.2888,128.9610,6,06,16,90,895.646235,4.069642,895.620430,893.740450,898.301030,884.04375,905.3379
11369,ldaps,surface_0_sp,6,37.2888,128.9610,6,06,17,90,895.683905,4.056831,895.562450,893.734200,898.417850,884.50270,905.4683
11367,ldaps,surface_0_sp,6,37.2888,128.9610,6,06,15,90,895.692698,4.039023,895.546950,893.620700,898.073963,883.31930,905.3409
11370,ldaps,surface_0_sp,6,37.2888,128.9610,6,06,18,90,895.795453,4.036726,895.759300,893.581450,898.426925,885.61110,905.7561
11366,ldaps,surface_0_sp,6,37.2888,128.9610,6,06,14,90,895.859394,4.078218,895.725225,893.908350,898.174085,883.08664,905.5120
12808,ldaps,surface_0_sp,11,37.2750,128.9604,6,06,16,90,895.875902,4.060239,895.857930,893.912850,898.497280,884.31875,905.5379
12807,ldaps,surface_0_sp,11,37.2750,128.9604,6,06,15,90,895.919309,4.034915,895.771950,893.809425,898.279775,883.61930,905.5609
12809,ldaps,surface_0_sp,11,37.2750,128.9604,6,06,17,90,895.921794,4.043231,895.797450,893.957050,898.620925,884.91270,905.6933
12810,ldaps,surface_0_sp,11,37.2750,128.9604,6,06,18,90,896.036675,4.021504,896.007300,893.956350,898.701925,885.93110,905.9661
11080,ldaps,surface_0_sp,5,37.2894,128.9437,6,06,16,90,896.051569,4.023310,896.065430,894.060525,898.523250,885.15375,905.6629


Highest pressure variability season-hour combinations:


,source,variable,grid_id,latitude,longitude,season,hour,count,mean_hpa,std_hpa,median_hpa,q25_hpa,q75_hpa,min_hpa,max_hpa
2960,ldaps,meanSea_0_prmsl,13,37.2740,128.9951,Autumn,8,273,1017.320075,6.250758,1017.453700,1013.782600,1021.863500,981.60870,1031.45160
2915,ldaps,meanSea_0_prmsl,13,37.2740,128.9951,Spring,11,276,1014.700871,6.246413,1015.367350,1010.895450,1018.571175,988.32770,1030.43164
2480,ldaps,meanSea_0_prmsl,8,37.2878,128.9958,Autumn,8,273,1017.344260,6.245906,1017.456200,1013.769500,1021.873800,981.77490,1031.52060
2435,ldaps,meanSea_0_prmsl,8,37.2878,128.9958,Spring,11,276,1014.728474,6.241698,1015.402800,1010.916095,1018.589888,988.27460,1030.44195
2339,ldaps,meanSea_0_prmsl,7,37.2883,128.9784,Spring,11,276,1014.742696,6.228272,1015.449400,1010.886387,1018.659550,988.64270,1030.43760
2819,ldaps,meanSea_0_prmsl,12,37.2745,128.9778,Spring,11,276,1014.728211,6.227274,1015.488250,1010.828410,1018.635215,988.76960,1030.42320
2864,ldaps,meanSea_0_prmsl,12,37.2745,128.9778,Autumn,8,273,1017.347372,6.224794,1017.466600,1013.757340,1021.844840,981.80740,1031.23000
3203,ldaps,meanSea_0_prmsl,16,37.2607,128.9771,Spring,11,276,1014.726233,6.224215,1015.481150,1010.802950,1018.567650,988.83770,1030.41600
2914,ldaps,meanSea_0_prmsl,13,37.2740,128.9951,Spring,10,276,1014.986403,6.224046,1015.425325,1011.235550,1018.889250,989.42800,1030.89516
2384,ldaps,meanSea_0_prmsl,7,37.2883,128.9784,Autumn,8,273,1017.362662,6.223906,1017.486640,1013.743300,1021.859800,981.91120,1031.33940


Output files:
- pressure_long: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\pressure_grid_hourly_long.csv
- month_hour_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\pressure_month_hour_grid_summary.csv
- season_hour_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\pressure_season_hour_grid_summary.csv
- overall_grid_summary: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\pressure_overall_grid_summary.csv
Plot directory: C:\Users\심현석\Documents\BARAM 2026\outputs\pressure_grid_eda\plots
